# Understanding Normalization Fundamentals
## Layer Normalization vs. Batch Normalization Concepts

Before implementing **Layer Normalization** in the GPT architecture, this notebook provides intuition on how normalization affects activation distributions, sample means, and sample variances.

---

### Key Concepts:
1. **The Problem of Internal Covariate Shift**: As inputs pass through multiple linear and non-linear layers, their activation distributions can shift, making deep networks harder to train.
2. **Mean & Variance Calculation**:
   $$\mu = \frac{1}{d} \sum_{i=1}^d x_i, \quad \sigma^2 = \frac{1}{d} \sum_{i=1}^d (x_i - \mu)^2$$
3. **Standardization (Centering & Scaling)**:
   $$\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}$$
   This rescales activations so that they have zero mean ($\mu \approx 0$) and unit variance ($\sigma^2 \approx 1$).

---
## 1. Imports and Setup
We import PyTorch and basic utilities.

In [1]:
import math
import torch
import torch.nn as nn
import tiktoken
import matplotlib.pyplot as plt

torch.set_printoptions(sci_mode=False)

/home/lakshay/.config/matplotlib is not a writable directory


Matplotlib created a temporary cache directory at /tmp/matplotlib-z__ehn2c because there was an issue with the default path (/home/lakshay/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


---
## 2. Empirical Demonstration: Normalizing Layer Activations

Below, we pass a random batch through a linear layer with ReLU activation and compute sample statistics:

In [2]:
torch.manual_seed(123)

batch_example = torch.rand(2, 5)
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_example)

print("Layer output:\n", out)
mean = out.mean(dim=1, keepdim=True)
var = out.var(dim=1, keepdim=True)

print("Mean per sample:\n", mean)
print("Variance per sample:\n", var)

# Manually normalize across the feature dimension
normed = (out - mean) / torch.sqrt(var)
print("Normalized values:\n", normed)
print("Normalized mean (close to 0):\n", normed.mean(dim=1, keepdim=True))
print("Normalized variance (close to 1):\n", normed.var(dim=1, keepdim=True))

Layer output:
 tensor([[0.0000, 0.0000, 0.4091, 0.6587, 0.3914, 0.0000],
        [0.0000, 0.0000, 0.1902, 0.3182, 0.6486, 0.0000]],
       grad_fn=<ReluBackward0>)
Mean per sample:
 tensor([[0.2432],
        [0.1928]], grad_fn=<MeanBackward1>)
Variance per sample:
 tensor([[0.0799],
        [0.0670]], grad_fn=<VarBackward0>)
Normalized values:
 tensor([[-0.8603, -0.8603,  0.5869,  1.4698,  0.5242, -0.8603],
        [-0.7450, -0.7450, -0.0102,  0.4844,  1.7608, -0.7450]],
       grad_fn=<DivBackward0>)
Normalized mean (close to 0):
 tensor([[ 0.0000],
        [-0.0000]], grad_fn=<MeanBackward1>)
Normalized variance (close to 1):
 tensor([[1.],
        [1.]], grad_fn=<VarBackward0>)


---
## 3. Key Takeaways for Transformer Architectures

- **Batch Normalization (BatchNorm)** normalizes across the *batch dimension*. This works well for computer vision, but fails for NLP because text sequences have variable lengths and small/dynamic batch sizes.
- **Layer Normalization (LayerNorm)** normalizes across the *feature/channel dimension* for each token independently. This makes LayerNorm invariant to batch size and sequence length, which is why GPT and modern Transformers rely exclusively on LayerNorm.